# Kriged map distribution

**Yuxi Zhang** · Oct 25, 2026 · [@yuxi-research](https://github.com/yuxi-research)

📘 This notebook plots the distribution of kriged maps from both B2PK and P2PK maps.

⚙️ **Main Processing Steps (Calibration)**  



💬 **Notes**

📥 **Input**  

<!-- - `./output1_<farm_name>/soilsamples/`  
   - `<farm_name>_soilsamples_fillna.csv`: Sampled soil moisture and covariates for all dates and depths (standard intervals).  

- `settings.py`: Configuration file specifying the farm name, target CRS, folder structure, and file paths. -->

📤 **Output**  

<!-- - `./output2_<farm_name>/kriging/`: Contains kriged maps of soil moisture and covariates for each depth interval. -->


In [ ]:
%load_ext autoreload
%autoreload 2

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd
# from shapely.geometry import Point
import re
# import seaborn as sns
import xarray as xr
import rioxarray as rxr
import warnings
warnings.filterwarnings("ignore")

save_pdf = True


In [ ]:
# Path settings
from global_settings import *
farm_name = os.environ.get("FARM_NAME", "Amaroo")
dirs = ProjectPaths(farm_name=farm_name)
proj_crs = dirs.proj_crs
inset_loc = dirs.inset_loc
buffer_dis = BUFFER_DIS


In [ ]:
dir_out = dirs.lv3_map / 'analysis'
dir_out_f = dir_out / 'figures'
dir_out_f.mkdir(parents=True, exist_ok=True)

## Read all data
### Find kriged rover counts

In [ ]:
# Find all corrected counts files and dates
mode = 'b2pk'
regex = re.compile(rf'{farm_name}_(.*)_CountsCorr_grid_masked_{mode}.tif')
filenames_rover = [f for f in os.listdir(dirs.lv2_krig / f'kriged_{mode}') if regex.match(f)]
filenames_rover.sort()
dates_str = [regex.match(f).group(1) for f in filenames_rover]
filenames_rover

In [ ]:
# Parameters
try:
    date_str
except NameError:
    date_str = dates_str[0]
print(f'Processing date: {date_str}')

### Read all counts maps (kriged) for both modes

In [ ]:
# Read all counts maps (kriged) for both modes
ds_counts = {}
var_plot = 'CountsCorr'
for mode in ['b2pk', 'p2pk']:
    dir_krig = dirs.lv2_krig / f'kriged_{mode}'
    rxrdata = rxr.open_rasterio(dir_krig / f'{farm_name}_{date_str}_{var_plot}_grid_masked_{mode}.tif', 
                                mask_and_scale=True).squeeze(drop=True)[0]
    ds_counts[mode] = rxrdata

ds_counts = xr.Dataset(ds_counts)


### Read all soil moisture maps (kriged) for both modes

In [ ]:
# Read all counts maps (kriged) for both modes
ds_sm = {}

for mode in ['b2pk', 'p2pk']:

    try:
        varname = 'SMVol'
        dir_file = dirs.lv3_map  / f'maps_{mode}' / f'{farm_name}_{date_str}_{varname}_{mode}.tif'
        assert dir_file.exists(), f"File not found: {dir_file}"

    except Exception:
        print(f"File not found: {dir_file}. Trying SMGrav instead.")
        try:
            varname = 'SMGrav'
            dir_file = dirs.lv3_map  / f'maps_{mode}' / f'{farm_name}_{date_str}_{varname}_{mode}.tif'
            assert dir_file.exists(), f"File not found: {dir_file}"

        except Exception as e:
            print(f"File not found: {dir_file}. No valid SM file found for mode {mode} and date {date_str}.")

    # Get variable type for display
    var_type = varname[2:]
    # Read file
    rxrdata = rxr.open_rasterio(dir_file, mask_and_scale=True).squeeze(drop=True)
    ds_sm[mode] = rxrdata

# Concat
ds_sm = xr.Dataset(ds_sm)

#### Boundary

In [ ]:
# Read boundary
bound = gpd.read_file(os.path.join(dirs.lv1_shp, f'{farm_name}_boundary_epsg{proj_crs}.shp'))
bound = bound.to_crs(proj_crs)

### Convolute to block-scale using the same weighting function


In [ ]:
from shapely.geometry import Point, MultiPoint

In [ ]:
 # Equivalent to resolution of the kriged map
discretize_spacing = abs(ds_counts.rio.resolution()[0])
radius = 240
# def calc_wr(r):
#     #Schron 2015 Eq 5
#     return np.exp(-r/127)

def calc_wr(r, r86=radius):
    #Kohli 2026 Review Eq. 1.
    return np.exp(-2*r/r86)


In [ ]:
# Descritize each block into multiple points around centroid for kriging with weights.
def discretize_block(centroid, radius=radius, spacing=discretize_spacing):
    coords_X = []
    dis_X = []
    weights_X = []
    coords_mat = []
    weights_mat = []
    # Make sure to include 0
    interval = np.arange(- np.floor(radius/spacing) * spacing, 
                         (np.floor(radius/spacing)) * spacing +1, spacing)
    for i in interval:
        for j in interval:  
            dis_ij = np.sqrt(i**2 + j**2)
            weight = calc_wr(dis_ij)
            if (0 <= dis_ij <= radius) and (weight > 0):
                coords_X.append(Point(centroid.x + i, centroid.y + j))
                dis_X.append(dis_ij)
                weights_X.append(weight)
                weights_mat.append(weight)
            else:
                weights_mat.append(0)

            coords_mat.append(Point(centroid.x + i, centroid.y + j))

    # Create MultiPoint and array of coordinates
    coords_X = MultiPoint(coords_X)
    dis_X = np.array(dis_X)
    weights_X = np.array(weights_X) / np.sum(weights_X)
    weights_mat = np.array(weights_mat) / np.sum(weights_mat)
    weights_mat = weights_mat.reshape(len(interval), len(interval))
    # coords_mat = coords_mat.reshape(len(interval), len(interval))
    return coords_X, dis_X, weights_X, weights_mat

# Apply discretization to a test block to obtain weights
coords_test, _, weights, weights_mat = discretize_block(Point(0, 0))


In [ ]:
# Visualize weighting function
for geom, weight in zip(coords_test.geoms, weights):
    x = geom.x
    y = geom.y
    z = weight
# Plot 3D weights
fig = plt.figure(figsize=(12, 6))
ax1 = fig.add_subplot(121, projection='3d')
# Create a surface plot
X = np.array([geom.x for geom in coords_test.geoms])
Y = np.array([geom.y for geom in coords_test.geoms])
Z = np.array([weight for weight in weights])

# ax1.scatter(xs=X, ys=Y, zs=Z, c='magenta', marker='o')
if len(X) > 0:
    dx = dy = discretize_spacing
    dz = Z
    ax1.bar3d(X.flatten()-dx/2, Y.flatten()-dy/2, np.zeros_like(Z), dx, dy, dz, color='pink', alpha=0.8)
    xmin, xmax = ax1.get_xlim()
    ymin, ymax = ax1.get_ylim()
    # Project a shadow on the XZ plane
    ax1.bar3d(X.flatten(), np.full_like(Y.flatten(), ymax), np.zeros_like(Z), dx, 0.01, dz, color='lightgrey', alpha=1)

    # Project a shadow on the YZ plane
    ax1.bar3d(np.full_like(X.flatten(), xmin), Y.flatten(), np.zeros_like(Z), 0.01, dy, dz, color='lightgrey', alpha=1)

    ax1.set_xlim(xmin, xmax)
    ax1.set_ylim(ymin, ymax)
    # ax1.view_init(elev=30, azim=45)
    # ax1.plot_trisurf(X, Y, Z, cmap='viridis', edgecolor='none', alpha=0.8)
else:
    print("Discretization resulted in only one point; skipping weight visualization.")

ax1.set_xlabel('X')
ax1.set_ylabel('Y')
ax1.set_zlabel('Weight')
plt.tight_layout()
# ax2 = fig.add_subplot(122)
# sc = ax2.scatter(X, Z, c= 'deeppink', s=100)



In [ ]:
def plot_violin(ds, var_plot, var_show=None, unit=''):
    if var_show is None:
        var_show = var_plot
    ds['b2pk_agg'] = ds['b2pk'].copy()
    kernel = weights_mat
    quantiles = [0, 0.01, 0.25, 0.5, 0.75, 0.99, 1]

    from scipy.signal import convolve2d
    # Convolve the kriged map with the kernel to get the aggregated map
    ds['b2pk_agg'].data = convolve2d(ds['b2pk'].data, kernel, mode='same', boundary='symm')

    arrs = {}
    df_quantile = []
    for mode in ['b2pk', 'b2pk_agg', 'p2pk']:
        arr = ds[mode].data.ravel()
        arr = arr[~np.isnan(arr)]
        arrs[mode] = arr
        df_quantile.append(np.quantile(arrs[mode], quantiles))

    df_quantile = pd.DataFrame(df_quantile, index=['b2pk', 'b2pk_agg', 'p2pk'], columns=[f'{int(q*100)}%' for q in quantiles])

    import seaborn as sns
    # Violin plot
    fig = plt.figure(figsize=(5.3, 4))
    sns.violinplot([arrs['b2pk'], arrs['b2pk_agg'], arrs['p2pk']], palette='Set1', alpha=0.8)
    ax = plt.gca()
    ax.set_xticklabels(['B2PK\n(point)', 'B2PK Agg.\n(block)', 'P2PK\n(block)'])
    ax.set_ylabel(f'{var_show} ({unit})')
    ax.set_title(f'{farm_name} {date_str}')
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    outpath = os.path.join(dir_out_f, f'{farm_name}_{date_str}_{var_plot}_violin.png')
    plt.savefig(outpath)
    if save_pdf:
        plt.savefig(outpath.replace('.png', '.pdf'))
    print('Figure saved to:', outpath)
    plt.show()
    return df_quantile

In [ ]:
# Plotting
df_counts = plot_violin(ds_counts, 'CountsCorr', 'Corr. Counts', unit='cpm')

unit_sm = r'$\mathrm{m}^3\ \mathrm{m}^{-3}$' if varname == 'SMVol' else r'$\mathrm{g}\ \mathrm{g}^{-1}$'
df_sm = plot_violin(ds_sm, varname, f'SM {var_type}', unit=unit_sm)

In [ ]:
# # Export quantiles to CSV
# df_counts.to_csv(os.path.join(dir_out, f'{farm_name}_{date_str}_{var_plot}_quantiles.csv'))
# df_sm.to_csv(os.path.join(dir_out, f'{farm_name}_{date_str}_{varname}_quantiles.csv'))